In [1]:
import numpy as np                              # 导入NumPy数值计算库，用于矩阵运算和科学计算
# ============================================================  # 分隔线，开始函数文档注释块
# 函数名: truss3d_element_stiffness                            # 说明函数名称：三维桁架单元刚度矩阵计算
# 功能: 计算三维杆单元长度、方向余弦、6×6全局刚度矩阵        # 说明函数功能：几何+材料→刚度矩阵
# 输入:                                                        # 输入参数说明开始
#   x1, x2 : 节点1、节点2坐标 [x, y, z]                      # x1,x2是两个节点的三维坐标列表
#   E      : 弹性模量 (Pa)                                    # 材料弹性模量，单位帕斯卡
#   A      : 横截面积 (m²)                                    # 杆件横截面面积，单位平方米
# 输出:                                                        # 输出参数说明开始
#   L      : 单元长度 (m)                                     # 两节点间的空间直线距离
#   c      : 方向余弦 [cx, cy, cz]                            # 局部坐标轴在全局坐标系中的方向余弦
#   Ke     : 6×6 全局刚度矩阵                                 # 6自由度（每节点3个）的单元刚度矩阵
# ============================================================  # 分隔线，结束文档注释块
def truss3d_element_stiffness(x1, x2, E, A):                   # 定义函数，接收两个节点坐标、弹性模量、截面积
    # 转为浮点数组，避免整数运算                              # 注释：类型转换目的
    x1 = np.array(x1, dtype=float)                             # 将节点1坐标转为NumPy浮点数组，防止整数除法错误
    x2 = np.array(x2, dtype=float)                             # 将节点2坐标转为NumPy浮点数组，确保数值精度
    # 坐标差                                                   # 注释：计算三个方向的坐标差值
    dx = x2[0] - x1[0]                                         # 计算x方向坐标差：节点2的x减去节点1的x
    dy = x2[1] - x1[1]                                         # 计算y方向坐标差：节点2的y减去节点1的y
    dz = x2[2] - x1[2]                                         # 计算z方向坐标差：节点2的z减去节点1的z
    # 单元长度                                                 # 注释：计算空间杆件的实际长度
    L = np.sqrt(dx**2 + dy**2 + dz**2)                         # 用欧几里得距离公式计算三维空间直线长度
    # 防退化: 节点重合时报错                                   # 注释：检查退化情况
    if L < 1e-12:                                              # 判断长度是否接近0（两节点重合或距离极小）
        raise ValueError("错误：两节点重合，单元退化，无法计算！") # 若重合则抛出异常，终止计算，提示建模错误
    # 方向余弦                                                 # 注释：计算局部坐标轴的方向余弦
    cx = dx / L                                                # 计算局部x轴与全局x轴夹角的余弦值（x投影/长度）
    cy = dy / L                                                # 计算局部x轴与全局y轴夹角的余弦值（y投影/长度）
    cz = dz / L                                                # 计算局部x轴与全局z轴夹角的余弦值（z投影/长度）
    # 刚度系数 EA/L                                            # 注释：计算杆件轴向刚度系数
    EA_L = E * A / L                                           # 弹性模量×截面积÷长度，杆单元的基本刚度参数
    # 6×6 三维杆单元刚度矩阵                                   # 注释：构建完整的6自由度刚度矩阵
    Ke = np.array([                                            # 用NumPy数组创建6×6矩阵，每行一个列表
        [ cx**2,  cx*cy,  cx*cz, -cx**2, -cx*cy, -cx*cz],      # 第1行：节点1的x方向平衡方程系数
        [ cx*cy,  cy**2,  cy*cz, -cx*cy, -cy**2, -cy*cz],      # 第2行：节点1的y方向平衡方程系数
        [ cx*cz,  cy*cz,  cz**2, -cx*cz, -cy*cz, -cz**2],      # 第3行：节点1的z方向平衡方程系数
        [-cx**2, -cx*cy, -cx*cz,  cx**2,  cx*cy,  cx*cz],      # 第4行：节点2的x方向平衡方程系数
        [-cx*cy, -cy**2, -cy*cz,  cx*cy,  cy**2,  cy*cz],      # 第5行：节点2的y方向平衡方程系数
        [-cx*cz, -cy*cz, -cz**2,  cx*cz,  cy*cz,  cz**2]       # 第6行：节点2的z方向平衡方程系数
    ]) * EA_L                                                  # 整个矩阵乘以EA/L，得到实际刚度值
    return L, np.array([cx, cy, cz]), Ke                       # 返回三个值：长度、方向余弦数组、刚度矩阵
# ============================================================  # 分隔线，开始第二个函数文档注释
# 函数名: truss3d_element_stress                               # 说明函数名称：三维桁架单元应力计算
# 功能: 由节点位移计算单元应变、应力、轴力                    # 说明函数功能：位移→力学响应的后处理
# 输入:                                                        # 输入参数说明开始
#   x1, x2, E, A : 单元几何与材料参数                          # 与刚度函数相同的材料几何参数
#   de           : 节点位移向量 [u1, v1, w1, u2, v2, w2]      # 6个自由度的节点位移列向量
# 输出:                                                        # 输出参数说明开始
#   epsilon : 轴向应变                                         # 杆件的轴向拉伸/压缩应变
#   sigma   : 轴向应力 (Pa)                                   # 根据胡克定律计算的应力
#   N       : 轴力 (N)                                        # 轴力=应力×面积
# ============================================================  # 分隔线，结束文档注释
def truss3d_element_stress(x1, x2, E, A, de):                  # 定义应力计算函数，额外接收位移向量
    L, c, _ = truss3d_element_stiffness(x1, x2, E, A)          # 调用刚度函数获取几何参数，下划线忽略刚度矩阵
    cx, cy, cz = c                                             # 将方向余弦数组解包为三个独立变量
    # 应变-位移矩阵 (行向量)                                   # 注释：构建1×6的应变-位移关系矩阵
    B = np.array([-cx, -cy, -cz, cx, cy, cz]) / L              # B矩阵：负号对应节点1，正号对应节点2，除以长度
    # 应变、应力、轴力                                          # 注释：依次计算三个力学量
    epsilon = B @ de                                           # 矩阵乘法：B（1×6）× de（6×1）= 标量应变值
    sigma   = E * epsilon                                      # 胡克定律：应力 = 弹性模量 × 应变
    N       = sigma * A                                        # 轴力公式：轴力 = 应力 × 横截面积
    return epsilon, sigma, N                                   # 返回三个计算结果
# ============================================================  # 分隔线，开始第三个函数文档注释
# 函数名: check_matrix_properties                              # 说明函数名称：矩阵性质检查
# 功能: 检查刚度矩阵的对称性、奇异性、半正定性                # 说明函数功能：验证数学特性
# 输入: Ke: 6×6刚度矩阵                                        # 接收刚度矩阵作为输入
# 输出: sym, singular, semidef, det, eigvals                   # 返回五个检查结果
# ============================================================  # 分隔线，结束文档注释
def check_matrix_properties(Ke):                               # 定义矩阵检查函数
    # 对称性                                                   # 注释：检查矩阵是否对称
    sym = np.allclose(Ke, Ke.T, atol=1e-10)                    # 比较矩阵与其转置，在1e-10容差内判断是否相等
    # 行列式                                                   # 注释：计算矩阵行列式
    det = np.linalg.det(Ke)                                    # 用NumPy线性代数模块计算行列式值
    # 特征值                                                   # 注释：计算矩阵特征值
    eigvals = np.linalg.eigvalsh(Ke)                           # 用eigvalsh计算实对称矩阵的特征值（高效算法）
    # 奇异性                                                   # 注释：判断矩阵是否奇异
    singular = abs(det) < 1e-10                                # 行列式绝对值小于阈值则判定为奇异（存在刚体模式）
    # 半正定性                                                 # 注释：判断矩阵是否半正定
    semidef = np.all(eigvals >= -1e-10)                        # 所有特征值非负（允许微小数值误差）则为半正定
    #修正最小特征值                                            # 注释：可能是预留功能或笔误
    return sym, singular, semidef, det, eigvals                 # 返回五个检查结果元组
# ============================================================  # 分隔线，开始测试函数1
# 函数名: test_case1                                           # 说明函数名称：测试算例1
# 功能: 算例1: 沿x轴一维杆单元                                # 说明函数功能：最简单情况验证
# ============================================================  # 分隔线，结束文档注释
def test_case1():                                              # 定义测试函数1
    print("=" * 60)                                            # 打印60个等号作为分隔线
    print("                算例1: 沿x轴一维杆单元")              # 打印居中的标题文字
    print("=" * 60)                                            # 打印60个等号作为分隔线
    # 单元参数                                                 # 注释：定义算例1的具体参数
    x1 = [0, 0, 0]                                             # 节点1坐标：原点
    x2 = [2, 0, 0]                                             # 节点2坐标：x轴上2米处
    E  = 200e9                                                 # 弹性模量：200GPa（钢材典型值）
    A  = 1.0e-4                                                # 截面积：0.0001平方米（1平方厘米）
    de = [0, 0, 0, 1.0e-3, 0, 0]                               # 位移向量：节点1固定，节点2沿x位移1mm
    # 计算刚度                                                 # 注释：调用核心计算函数
    L, c, Ke = truss3d_element_stiffness(x1, x2, E, A)          # 计算几何参数和刚度矩阵
    eps, sig, N = truss3d_element_stress(x1, x2, E, A, de)      # 计算应变、应力、轴力
    # 输出几何                                                 # 注释：打印几何计算结果
    print(f"单元长度 L = {L:.2f} m")                           # 格式化输出长度，保留2位小数
    print(f"方向余弦 c = [{c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f}]") # 格式化输出方向余弦，保留4位小数
    print()                                                    # 打印空行，增加可读性
    # 输出刚度矩阵 (仅非零项)                                  # 注释：提取有效自由度对应的子矩阵
    print("刚度矩阵 (仅非零项): ")                              # 打印说明文字
    Ke_2x2 = np.array([                                        # 创建2×2子矩阵，只保留x方向自由度
        [Ke[0,0], Ke[0,3]],                                    # 第1行：u1对应的刚度系数，u2耦合系数
        [Ke[3,0], Ke[3,3]]                                     # 第2行：u2耦合系数，u2自刚度系数
    ])                                                         # 结束子矩阵定义
    print(Ke_2x2)                                              # 打印2×2子矩阵
    print()                                                    # 打印空行
    # 输出力学结果                                             # 注释：打印力学计算结果
    print(f"轴向应变    ε = {eps:.4e}")                         # 科学计数法输出应变，4位有效数字
    print(f"轴向应力    σ = {sig/1e6:.2f} MPa")                 # 应力转换为MPa，保留2位小数
    print(f"轴力        N = {N:.2e} N")                         # 科学计数法输出轴力，2位小数
    print()                                                    # 打印空行
    # 矩阵性质 (每项单独一行)                                  # 注释：检查并打印刚度矩阵性质
    sym, sing, semid, det, eigv = check_matrix_properties(Ke)   # 调用检查函数，解包五个返回值
    print("刚度矩阵性质: ")                                     # 打印标题
    print(f"  对称性    : {sym}")                                # 打印对称性检查结果（True/False）
    print(f"  奇异性    : {sing}")                                # 打印奇异性检查结果（应为True）
    print(f"  半正定性  : {semid}")                              # 打印半正定性检查结果（应为True）
    print(f"  行列式    : {det:.2e}")                            # 科学计数法打印行列式
    print(f"  最小特征值 : {np.min(eigv):.2e}")                   # 打印最小特征值（应为0，对应刚体模式）
    print()                                                    # 打印空行
# ============================================================  # 分隔线，开始测试函数2
# 函数名: test_case2                                           # 说明函数名称：测试算例2
# 功能: 算例2: 空间任意方向杆单元                             # 说明函数功能：三维空间一般情况
# ============================================================  # 分隔线，结束文档注释
def test_case2():                                              # 定义测试函数2
    print("=" * 60)                                            # 打印60个等号分隔线
    print("                算例2: 空间任意方向杆单元")           # 打印居中的标题
    print("=" * 60)                                            # 打印60个等号分隔线
    # 单元参数                                                 # 注释：定义空间斜杆参数
    x1 = [0, 0, 0]                                             # 节点1坐标：原点
    x2 = [1, 2, 2]                                             # 节点2坐标：(1,2,2)，长度应为3米
    E  = 210e9                                                 # 弹性模量：210GPa
    A  = 2.0e-4                                                # 截面积：0.0002平方米
    de = [0, 0, 0, 1.0e-3, 2.0e-3, 2.0e-3]                    # 节点2位移沿(1,2,2)方向，与杆轴同向
    # 计算刚度                                                 # 注释：调用计算函数
    L, c, Ke = truss3d_element_stiffness(x1, x2, E, A)          # 计算几何和刚度
    eps, sig, N = truss3d_element_stress(x1, x2, E, A, de)      # 计算力学响应
    # 输出几何                                                 # 注释：打印几何结果
    print(f"单元长度 L = {L:.2f} m")                           # 输出长度，应为3.00m
    print(f"方向余弦 c = [{c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f}]") # 输出方向余弦，应为[0.3333,0.6667,0.6667]
    print()                                                    # 空行
    # 输出刚度矩阵 (完整6×6)                                   # 注释：输出完整刚度矩阵
    print("刚度矩阵 (6x6): ")                                  # 打印说明
    print(Ke)                                                  # 打印完整的6×6矩阵
    print()                                                    # 空行
    # 输出力学结果                                             # 注释：打印力学量
    print(f"轴向应变    ε = {eps:.4e}")                         # 输出应变
    print(f"轴向应力    σ = {sig/1e6:.2f} MPa")                 # 输出应力（MPa）
    print(f"轴力        N = {N:.2e} N")                         # 输出轴力
    print()                                                    # 空行
    # 矩阵性质 (每项单独一行)                                  # 注释：检查矩阵性质
    sym, sing, semid, det, eigv = check_matrix_properties(Ke)   # 调用检查函数
    print("刚度矩阵性质: ")                                     # 打印标题
    print(f"  对称性    : {sym}")                                # 打印对称性
    print(f"  奇异性    : {sing}")                                # 打印奇异性
    print(f"  半正定性  : {semid}")                              # 打印半正定性
    print(f"  行列式    : {det:.2e}")                            # 打印行列式
    print(f"  最小特征值 : {np.min(eigv):.2e}")                   # 打印最小特征值
    print()                                                    # 空行
    # 刚体位移验证                                             # 注释：验证刚体运动不产生内力
    print("刚体位移验证: ")                                     # 打印标题
    de_rigid = [0.1, 0.2, 0.3, 0.1, 0.2, 0.3]                  # 两节点相同平移，无相对变形
    eps_r, sig_r, N_r = truss3d_element_stress(x1, x2, E, A, de_rigid) # 计算刚体位移对应的力学量
    print(f"  应变 : {eps_r:.2e}")                               # 应变应为0（或接近0）
    print(f"  应力 : {sig_r:.2e} Pa")                            # 应力应为0
    print(f"  轴力 : {N_r:.2e} N")                              # 轴力应为0
    print()                                                    # 空行
# ============================================================  # 分隔线，开始验证函数
# 函数名: verify_stiffness_meaning                             # 说明函数名称：物理意义验证
# 功能: 验证刚度矩阵物理意义: 单位位移->力向量=矩阵对应列     # 核心原理：K_ij = 单位位移j引起的力i
# ============================================================  # 分隔线，结束文档注释
def verify_stiffness_meaning():                                # 定义验证函数
    print("=" * 60)                                            # 打印分隔线
    print("                刚度矩阵物理意义验证")                # 打印标题
    print("=" * 60)                                            # 打印分隔线
    x1 = [0,0,0]                                               # 节点1坐标
    x2 = [1,2,2]                                               # 节点2坐标
    E  = 210e9                                                 # 弹性模量
    A  = 2.0e-4                                                # 截面积
    _, _, Ke = truss3d_element_stiffness(x1, x2, E, A)          # 只取刚度矩阵，忽略长度和方向余弦
    for j in [1, 2, 3]:                                        # 循环遍历自由度1,2,3（Python索引）
        de = np.zeros(6)                                        # 创建6维零向量作为单位位移基底
        de[j] = 1.0                                             # 第j个自由度设为1，其余为0
        Fe = Ke @ de                                            # 矩阵乘法：刚度矩阵×单位位移=节点力向量
        col = Ke[:, j]                                          # 提取刚度矩阵的第j列（所有行，第j列）
        print(f"自由度 j={j} 单位位移")                          # 打印当前测试的自由度编号
        print(f"力向量 Fe : {Fe}")                               # 打印计算得到的力向量
        print(f"刚度矩阵第{j}列 : {col}")                        # 打印矩阵第j列
        print(f"是否相等 : {np.allclose(Fe, col)}")              # 比较两者是否相等（验证K_ij定义）
        print()                                                    # 空行
# ============================================================  # 分隔线，主程序入口
# 主程序入口                                                   # 说明：以下代码在直接运行时执行
# ============================================================  # 分隔线
if __name__ == "__main__":                                     # Python惯用法：判断是否为直接运行（非导入）
    test_case1()                                               # 调用测试1：一维杆单元验证
    test_case2()                                               # 调用测试2：空间斜杆验证
    verify_stiffness_meaning()                                 # 调用验证：刚度矩阵物理意义



                算例1: 沿x轴一维杆单元
单元长度 L = 2.00 m
方向余弦 c = [1.0000, 0.0000, 0.0000]

刚度矩阵 (仅非零项): 
[[ 10000000. -10000000.]
 [-10000000.  10000000.]]

轴向应变    ε = 5.0000e-04
轴向应力    σ = 100.00 MPa
轴力        N = 1.00e+04 N

刚度矩阵性质: 
  对称性    : True
  奇异性    : True
  半正定性  : True
  行列式    : 0.00e+00
  最小特征值 : 0.00e+00

                算例2: 空间任意方向杆单元
单元长度 L = 3.00 m
方向余弦 c = [0.3333, 0.6667, 0.6667]

刚度矩阵 (6x6): 
[[ 1555555.55555556  3111111.11111111  3111111.11111111 -1555555.55555556
  -3111111.11111111 -3111111.11111111]
 [ 3111111.11111111  6222222.22222222  6222222.22222222 -3111111.11111111
  -6222222.22222222 -6222222.22222222]
 [ 3111111.11111111  6222222.22222222  6222222.22222222 -3111111.11111111
  -6222222.22222222 -6222222.22222222]
 [-1555555.55555556 -3111111.11111111 -3111111.11111111  1555555.55555556
   3111111.11111111  3111111.11111111]
 [-3111111.11111111 -6222222.22222222 -6222222.22222222  3111111.11111111
   6222222.22222222  6222222.22222222]
 [-3111111.11111111 -6222